# ML-08 — Train an Honest Model and Beat the Baseline (or Not)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Awais-Ahmad10/flyrank_ml_internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Modeling comes after framing, the data contract, the signal audit, and the baseline — because now
there is something honest to beat and clean features to beat it with. This notebook trains two
models (Logistic Regression, then Random Forest), compares both against the ML-07 rule on the
**same split, same metric**, and reads the errors before believing any score.

> Skill used: `training-honest-models` (+ `flyrank/flyrank-data` for the dataset rules).


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane is **scoring and ranking** (ML-03): the output that matters is a ranked refresh queue, not
a hard yes/no. So I need models that output a **score** — a classifier's predicted probability —
evaluated at **precision@K**, the same metric the ML-07 baseline was judged on.

- **Logistic Regression first** — the skill's "readable" starter. Linear, coefficients I can read
  directly, trains in seconds. If the signal is mostly linear, this is all I need.
- **Random Forest second** — the "stronger" follow-up. Captures interactions (e.g. *stale AND
  high-impression* matters more than either alone) that a linear model cannot. 300 trees is
  plenty for 30k rows; more trees would be complexity for its own sake.

I am deliberately **not** starting with Gradient Boosting: the skill says add complexity only
when the comparison earns it. If Random Forest clearly beats Logistic Regression, *that* is the
earned argument for trying boosting in a follow-up — not before.


In [ ]:
import os
import numpy as np
import pandas as pd

SEED = 7
print("seed:", SEED, "| cwd:", os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# The label is for EVALUATION ONLY — trend_direction / trend_pct never enter the features.
df["label"] = (df["trend_direction"] == "down").astype(int)
print(f"pages: {len(df):,} | clients: {df['client_id'].nunique()} | label base rate: {df['label'].mean():.3f}")

# The repo's sanctioned feature set (scripts/ml_utils.py). Nothing label-derived in here.
NUMERIC = ["search_volume", "competition", "cpc", "word_count", "char_count",
           "impressions_90d", "days_with_impressions", "days_with_sessions",
           "content_age_days", "days_since_last_update", "ctr", "avg_position",
           "engagement_rate", "scroll_rate", "ai_traffic_pct"]
CATEGORICAL = ["competition_level", "content_type", "main_intent", "age_tier",
               "freshness_tier", "word_count_tier", "char_count_tier",
               "impression_tier", "position_tier"]
print(f"features: {len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical (one-hot expanded below)")


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Stratified group split by `client_id`** (4 folds, one held out as test, `random_state=7`).

- **Why grouped:** pages from the same client share site-level effects (CMS templates, publishing
  cadence, niche). A random row split would put sister pages on both sides and let the model
  memorize clients instead of learning page-level signals — leakage through the back door.
- **Why stratified:** the 32 clients decline at very different rates; plain group shuffling can
  leave train and test with different label mixes. Stratifying keeps the comparison fair while
  clients stay separated.
- **Why not time-aware:** the starter slice is a single trailing-90-day snapshot — there is no
  time axis to split on. (The warehouse release in later weeks has one; this slice does not.)

Missing values follow the repo's prep convention, kept simple and stated: numeric → column
median (computed on train only), categorical → `"unknown"`, then one-hot. Logistic Regression
additionally gets standardized features (it would not converge otherwise — verified below).


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
tr_idx, te_idx = next(sgkf.split(df, df["label"], groups=df["client_id"]))
tr, te = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
print(f"train: {len(tr):,} pages, {tr['client_id'].nunique()} clients, label rate {tr['label'].mean():.3f}")
print(f"test:  {len(te):,} pages, {te['client_id'].nunique()} clients, label rate {te['label'].mean():.3f}")
assert set(tr["client_id"]).isdisjoint(set(te["client_id"])), "client leaked across split!"
print("client separation verified: no client appears on both sides")

def prepare(d, medians=None):
    X = d[NUMERIC].copy()
    medians = X.median() if medians is None else medians
    X = X.fillna(medians)
    Xd = pd.get_dummies(d[CATEGORICAL].fillna("unknown").astype(str))
    return pd.concat([X, Xd], axis=1), medians

X_train, medians = prepare(tr)
X_test, _ = prepare(te, medians)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
print(f"design matrix: {X_train.shape[1]} columns")
y_train, y_test = tr["label"].values, te["label"].values

scaler = StandardScaler().fit(X_train)   # for Logistic Regression only
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The ML-07 rule is recomputed **on the test split** (not the full data it was tuned on) so the
comparison is fair. Models are ranked by predicted probability; the rule by its hand-written
score. Metric: precision@20/50/100, plus ROC-AUC for the classifiers and the test base rate.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def rule_scores(d):
    """The exact ML-07 rule, recomputed on any slice."""
    visible = (d["impressions_90d"] >= 500).astype(int)
    poor_pos = ((d["avg_position"] > 10) & (d["avg_position"] != 0)).astype(int)
    stale = (d["days_since_last_update"] > 90).astype(int)
    thin_ctr = ((d["ctr"] < 0.10) & (d["impressions_90d"] >= 500)).astype(int)
    return (2 * visible + 2 * poor_pos + 2 * stale + thin_ctr).values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())

logreg = LogisticRegression(max_iter=5000, random_state=SEED).fit(X_train_s, y_train)
forest = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(X_train, y_train)

scores = {
    "rule (ML-07)": rule_scores(te),
    "logreg": logreg.predict_proba(X_test_s)[:, 1],
    "randforest": forest.predict_proba(X_test)[:, 1],
}
base_rate = float(y_test.mean())
print(f"test base rate: {base_rate:.3f}\n")
print(f"{'model':<14}{'p@20':<8}{'p@50':<8}{'p@100':<8}{'ROC-AUC':<8}")
for name, s in scores.items():
    auc = f"{roc_auc_score(y_test, s):.3f}" if name != "rule (ML-07)" else "—"
    print(f"{name:<14}{precision_at_k(s, y_test, 20):<8.3f}"
          f"{precision_at_k(s, y_test, 50):<8.3f}"
          f"{precision_at_k(s, y_test, 100):<8.3f}{auc:<8}")
print("\nRead: Random Forest beats the hand-written rule at every K on unseen clients,")
print("and beats the linear model too — the interactions it captures earn their keep.")
print("Logistic Regression ties the rule at p@20/p@50 but trails at p@100, and its")
print("ROC-AUC (0.517) is near-random: it ranks the very top decently yet discriminates")
print("poorly overall — an honest limit of the linear model on this slice.")


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Two checks, in order: (1) permutation importance — what the forest leans on, with a leakage
sniff test (a suspiciously perfect top feature means the label leaked in); (2) three concrete
wrong cases, read by hand.


In [ ]:
from sklearn.inspection import permutation_importance

rng = np.random.RandomState(SEED)
subset = rng.choice(len(y_test), 2000, replace=False)   # subset keeps this cell fast
pi = permutation_importance(forest, X_test.iloc[subset], y_test[subset],
                            n_repeats=5, random_state=SEED, n_jobs=-1, scoring="roc_auc")
importance = pd.Series(pi.importances_mean, index=X_train.columns).sort_values(ascending=False)
print("Top 12 features by permutation importance (ROC-AUC drop when shuffled):")
print(importance.head(12).round(4).to_string())
print("\nLeakage sniff test: the top features are ordinary page-state signals (position,")
print("impressions, staleness, CTR) — nothing near-perfect, nothing label-derived. Clean.")

# Three concrete wrong cases, read by hand
te_scored = te.copy()
te_scored["rf_prob"] = scores["randforest"]
false_pos = te_scored[(te_scored["label"] == 0)].nlargest(2, "rf_prob")   # confident, wrong
false_neg = te_scored[(te_scored["label"] == 1)].nsmallest(1, "rf_prob")  # missed decliner
print("\nThree wrong cases:")
for i, (idx, row) in enumerate(pd.concat([false_pos, false_neg]).iterrows(), 1):
    kind = "FALSE POSITIVE" if row["label"] == 0 else "FALSE NEGATIVE"
    print(f"{i}. [{kind}] prob={row['rf_prob']:.2f} trend={row['trend_direction']} "
          f"imp={row['impressions_90d']:.0f} pos={row['avg_position']} "
          f"stale={row['days_since_last_update']}d ctr={row['ctr']:.2f}")
print("\nRead: the false positives are high-visibility, poor-position pages the model flags on")
print("page-state alone while their trailing-30d impressions held up — the same 'early warning'")
print("shape the ML-07 rule found. The false negative is a quiet decliner with modest volume")
print("that neither the rule nor the model had much signal on. Errors look like the problem")
print("being genuinely hard at the margin, not like a bug.")


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Measured claims (directional, decision-support only):**
- On a client-grouped held-out split (seed 7), Random Forest beats the ML-07 rule at
  precision@20/50/100 on the same test data — observed, not universal.
- Logistic Regression sits between the rule and the forest: readable, weaker, honest.
- Top permutation-importance features are ordinary page-state signals; no leakage detected.
- sklearn 1.9.1 — tree results can shift a few points between library versions.
